# ML Validation

In [1]:
pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import classification_report, accuracy_score
import warnings
warnings.filterwarnings("ignore")

In [6]:
df = pd.read_csv('f1_clean_data.csv')
df = df.dropna(subset=['Finishing_Tier'])
df

,Race_ID,Driver,LapNumber,Finishing_Tier,GridPosition,Tire_Age,Fuel_Mass_Proxy,Degradation_x_Fuel,Tire_MEDIUM,Tire_SOFT
0,2023_Bahrain,VER,1.0,Podium,1.0,4.0,108.070175,-569.391815,0,1
1,2023_Bahrain,VER,2.0,Podium,1.0,5.0,106.140351,-498.135831,0,1
2,2023_Bahrain,VER,3.0,Podium,1.0,6.0,104.210526,-430.739496,0,1
3,2023_Bahrain,VER,4.0,Podium,1.0,7.0,102.280702,-367.202809,0,1
4,2023_Bahrain,VER,5.0,Podium,1.0,8.0,100.350877,-307.525772,0,1
...,...,...,...,...,...,...,...,...,...,...
2875,2023_Australia,PIA,50.0,No Points,16.0,42.0,13.508772,-1149.629808,0,0
2876,2023_Australia,PIA,51.0,No Points,16.0,43.0,11.578947,-1246.268560,0,0
2877,2023_Australia,PIA,52.0,No Points,16.0,44.0,9.649123,-1346.766962,0,0
2878,2023_Australia,PIA,53.0,No Points,16.0,45.0,7.719298,-1451.125012,0,0


In [7]:
#features and targets
tire_cols = [col for col in df.columns if col.startswith('Tire_') and col not in ['Tire_Age', 'Tire_Age.1']]

# ADDED GridPosition to the features list
base_features = ['GridPosition', 'Tire_Age', 'Fuel_Mass_Proxy'] + tire_cols

X = df[base_features]
y = df['Finishing_Tier']
groups = df['Race_ID']

In [9]:
#model = LogisticRegression(solver='saga', max_iter=1000)
model = LogisticRegression(solver='saga', max_iter=1000, class_weight='balanced')

# 4. Leave-One-Group-Out Cross-Validation (LOGO-CV)
logo = LeaveOneGroupOut()

all_y_true = []
all_y_pred = []


for train_index, test_index in logo.split(X, y, groups):
    X_train, X_test = X.iloc[train_index].copy(), X.iloc[test_index].copy()
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    # --- PREVENT DATA LEAKAGE
    train_mean_tire = X_train['Tire_Age'].mean()
    train_mean_fuel = X_train['Fuel_Mass_Proxy'].mean()

    X_train['Degradation_x_Fuel'] = (X_train['Tire_Age'] - train_mean_tire) * (X_train['Fuel_Mass_Proxy'] - train_mean_fuel)
    X_test['Degradation_x_Fuel'] = (X_test['Tire_Age'] - train_mean_tire) * (X_test['Fuel_Mass_Proxy'] - train_mean_fuel)
    

    model.fit(X_train, y_train)
    

    predictions = model.predict(X_test)
    

    all_y_true.extend(y_test)
    all_y_pred.extend(predictions)

#overall fit
print("\n--- CLASSIFICATION HIT RATIO (UNSEEN RACES) ---")
accuracy = accuracy_score(all_y_true, all_y_pred)
print(f"Overall Accuracy: {accuracy * 100:.2f}%\n")

print("--- CLASSIFICATION REPORT ---")
print(classification_report(all_y_true, all_y_pred))


--- CLASSIFICATION HIT RATIO (UNSEEN RACES) ---
Overall Accuracy: 62.36%

--- CLASSIFICATION REPORT ---
              precision    recall  f1-score   support

   No Points       0.74      0.67      0.70      1468
      Podium       0.57      0.89      0.69       429
Point Scorer       0.49      0.45      0.47       983

    accuracy                           0.62      2880
   macro avg       0.60      0.67      0.62      2880
weighted avg       0.63      0.62      0.62      2880

